In [ ]:
# Create benchmark_frag schema if it does not exist
spark.sql("CREATE SCHEMA IF NOT EXISTS benchmark_frag")
print("Schema benchmark_frag ready.")

# 00_setup_lh_frag — Fragmentation Experiment: Lakehouse Setup

Creates the `benchmark_frag` schema in `LH_01` with:
- **7 dimension tables** copied compactly from `benchmark_default`
- **`store_sales`** written with `maxRecordsPerFile=10000` → ~28.8K Parquet files of 10K rows each

**Prerequisites**: `benchmark_default` schema must be fully ingested (SF100).  
**Estimated runtime**: ~30–60 minutes (dominated by the `store_sales` write).  
**Run once** — idempotent (`CREATE OR REPLACE TABLE`).

In [ ]:
# Copy dimension tables compactly (no fragmentation needed for dimensions)
DIMENSION_TABLES = [
    "date_dim",
    "item",
    "store",
    "customer",
    "customer_demographics",
    "promotion",
    "household_demographics",
]

for table in DIMENSION_TABLES:
    print(f"Copying dimension: {table} ...", end=" ")
    spark.sql(f"""
        CREATE OR REPLACE TABLE benchmark_frag.{table}
        USING DELTA
        AS SELECT * FROM benchmark_default.{table}
    """)
    count = spark.sql(f"SELECT COUNT(*) AS n FROM benchmark_frag.{table}").collect()[0]["n"]
    print(f"done ({count:,} rows)")

print("\nAll dimension tables copied.")

In [ ]:
# Drop existing table to start with a clean Delta log and no leftover Parquet files.
spark.sql("DROP TABLE IF EXISTS benchmark_frag.store_sales")
print("Dropped benchmark_frag.store_sales (clean slate).")

# Write store_sales with maxRecordsPerFile=10000 to create ~28.8K small Parquet files.
# This is a single Spark job — all 288M rows are written in one pass, each output file
# limited to 10,000 rows. Expected runtime: ~30–60 min on SF100.

MAX_RECORDS_PER_FILE = 10000

print(f"Writing benchmark_frag.store_sales with maxRecordsPerFile={MAX_RECORDS_PER_FILE} ...")
print("This may take 30–60 minutes. Progress is not shown until the job completes.")

spark.conf.set("spark.sql.files.maxRecordsPerFile", str(MAX_RECORDS_PER_FILE))

(
    spark.table("benchmark_default.store_sales")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("maxRecordsPerFile", str(MAX_RECORDS_PER_FILE))
    .saveAsTable("benchmark_frag.store_sales")
)

# Reset to default (do not leave this set for subsequent notebook cells)
spark.conf.set("spark.sql.files.maxRecordsPerFile", "0")

print("Done: store_sales written.")

In [ ]:
# Verify row counts and file count
from pyspark.sql.functions import count as spark_count

print("=== Row counts ===")
tables = ["store_sales"] + DIMENSION_TABLES
for table in tables:
    row_count = spark.sql(f"SELECT COUNT(*) AS n FROM benchmark_frag.{table}").collect()[0]["n"]
    print(f"  benchmark_frag.{table}: {row_count:,}")

print("\n=== store_sales Delta file count ===")
detail = spark.sql("DESCRIBE DETAIL benchmark_frag.store_sales").collect()[0]
print(f"  numFiles: {detail['numFiles']:,}")
print(f"  sizeInBytes: {detail['sizeInBytes'] / 1024**3:.1f} GB")